# TCGA-KIRC • Laboratório Colab R934
## Protocolo de pré-análise auditável — SDD/TDD, GDC open-access e holdout lacrado

**Versão:** R934.1 · **Data do documento:** 21/10/2026 · **Estado inicial:** P0–P4 pendentes.  
**Questão:** a assinatura metabólica predefinida melhora a AUC cumulative/dynamic aos **60 meses** em relação ao modelo clínico (idade, sexo, estágio AJCC)? O objetivo é estimar **ΔAUC e IC95%**, inclusive se o resultado for nulo/negativo.

Este caderno implementa controles executáveis, os testes de contraprova RED→GREEN, consulta opcional ao **GDC `/files` somente metadados**, inventário e limites de acesso/volume, partição estratificada por paciente, diagnósticos de leakage e um roteiro de análise. **Não executa automaticamente análise de RNA-seq, instala pacotes, monta Drive ou transfere dados.**

> Escopo auditável: aqui P0/P1 recebem implementação completa *condicional*; P2/P3/P4 contêm contrato de execução e gates que permanecem fechados até que dados, métodos e revisão estejam disponíveis. Não transformar piloto Genomic Intelligence ou números do manuscrito em resultados clínicos.

### Mapa das etapas e controles

| Etapa | Implementação | Regra de parada |
|---|---|---|
| P0 | especificação congelada, testes positivos e contraprovas, aceite metodológico explícito | sem aceite, não se passa a P1 aprovado |
| P1 | GDC status/schema, metadados paginados, deduplicação, manifestos e orçamento ≤ 6 GiB | falha em acesso, checksum, workflow/release, origem amostral ou orçamento |
| P2 | downloader verificável **opcional** por MD5/SHA-256 e limite total | proíbe acesso controlado e arquivo não manifestado |
| P3 | split por paciente e testes de normalização/calibração em cenário sintético | **não libera modelagem prognóstica real** sem pipeline nested-CV/VST validado |
| P4 | estrutura de registro de AUC/IC/Brier/calibração, holdout único | permanece pendente até execução e revisão independentes |

**Importante:** o R934 enviado relata consulta *metadata-only* em 21/10/2026, 614 IDs únicos e 605 selecionados, **mas não disponibiliza um manifesto íntegro/verificável**. Não se presume que tais contagens sejam atuais ou suficientes para P1. A frase final “GDC ainda não foi consultado” é inconsistente com a abertura e foi corrigida nesta versão.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
import json, hashlib, re, os, statistics, unittest, io, contextlib
from collections import Counter

ROOT = Path('/content/R934_ccRCC') if Path('/content').exists() else Path.cwd() / 'R934_ccRCC'
for name in ('private_metadata','logs','reports','data_open','tests','models'):
    (ROOT/name).mkdir(parents=True, exist_ok=True)
def timestamp(): return datetime.now(timezone.utc).isoformat()
def sha256_obj(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()
def write_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2,sort_keys=True,ensure_ascii=False,default=str),encoding='utf8')
    return path
print('Diretório local (NÃO Google Drive):',ROOT)

## 1. Política de execução e aprovação explícita

Os parâmetros abaixo **não autorizam baixar dados**. P0 requer testes GREEN, escopo/metodologia congelados e revisão/aceite independente registrado. P1 exige manifesto reproduzível. A consulta metadata-only é desativada por padrão, apesar de ter sido relatada uma consulta exploratória anteriormente.

In [ ]:
PROJECT='TCGA-KIRC'
SEED=2026
GDC='https://api.gdc.cancer.gov'
MAX_AGGREGATE_BYTES=6*1024**3
REQUIRE_OPEN=True
REQUIRE_RELEASED=True
CONSULTAR_GDC_METADATA=False   # deliberadamente False na distribuição
EXECUTAR_DOWNLOADS=False       # permanece False até revisão P0 e P1
ACEITE_METODOLOGICO=''         # quando aplicável: preencher identificação/revisão e registrar em arquivo independente
CONFIRMACAO_DOWNLOAD=''       # para transferir: inserir exatamente 'AUTORIZO_DOWNLOAD_OPEN_6GIB'
EXPECTED_WORKFLOW='STAR - Counts'

GLYCOLYSIS=['SLC2A1','SLC2A3','HK1','HK2','HK3','GPI','PFKP','PFKM','PFKL','ALDOA','ALDOC','TPI1','GAPDH','PGK1','PGAM1','ENO1','ENO2','PKM','LDHA','PDK1','PDK3','SLC16A3']
TCA_OXPHOS=['PDHA1','PDHB','DLAT','DLD','CS','ACO2','IDH1','IDH2','IDH3A','OGDH','DLST','SUCLA2','SUCLG1','SDHA','SDHB','SDHC','SDHD','FH','MDH1','MDH2','PPARGC1A']
GLUCONEOGENESIS=['PCK1','PCK2']  # não confundir com TCA/OXPHOS
PANEL_APPROVED=False  # lista candidata requer conferência de fontes biológicas antes de congelar

SPEC=dict(project=PROJECT,seed=SEED,holdout_fraction=.30,primary_horizon_months=60,
          budget_bytes=MAX_AGGREGATE_BYTES,access='open',state='released',
          workflow=EXPECTED_WORKFLOW,outer_folds=5,outer_repeats=5,inner_folds=5,
          final_lambda='lambda.1se',sensitivity_lambda='lambda.min',
          auc_bootstrap=1000,min_bootstrap_convergence=.90,
          min_events_by_60m_holdout=10,min_censor_survival_at_60m=.10,
          glycolysis=GLYCOLYSIS,tca_oxphos=TCA_OXPHOS,gluconeogenesis=GLUCONEOGENESIS,
          auc_metric='cumulative_dynamic_IPCW',missingness_limit_per_covariate=.10)
SPEC_HASH=sha256_obj(SPEC)
write_json(ROOT/'reports'/'spec_R934_1.json',{'spec':SPEC,'sha256':SPEC_HASH,'saved_at_utc':timestamp()})
print('Hash do contrato SDD:',SPEC_HASH)

## 2. TDD — contraprovas RED→GREEN (inteiramente sintético, sem pacotes externos)

TDD **não avalia** se AUC, HR, MUI ou p-valor têm direção favorável. Os testes abaixo detectam erros de integridade. A função `reject()` lança um erro quando uma invariável é violada; contraprova RED é aprovada quando o erro esperado ocorre e a amostra positiva passa.

In [ ]:
class GateBlocked(RuntimeError): pass
class MetadataError(GateBlocked): pass

def require(condition,msg):
    if not condition: raise GateBlocked(msg)

def validate_counts(matrix):
    # matriz Python: genes x amostras; só TDD com valores sintéticos
    require(bool(matrix) and bool(matrix[0]),'matriz vazia')
    width=len(matrix[0]);require(all(len(row)==width for row in matrix),'dimensões irregulares')
    for row in matrix:
        for x in row:
            require(isinstance(x,(int,float)) and x>=0 and x!=float('inf') and x==x and float(x).is_integer(),
                    'contagens devem ser inteiros >= 0, finitos')
    require(all(sum(row[j] for row in matrix)>0 for j in range(width)),'biblioteca zerada')
    return True

def validate_patient_partition(train_ids,test_ids):
    require(len(train_ids)==len(set(train_ids)),'duplicatas em discovery')
    require(len(test_ids)==len(set(test_ids)),'duplicatas em holdout')
    require(not (set(train_ids)&set(test_ids)),'leakage: paciente em discovery e holdout')
    return True

def fit_train_scaler(x_train):
    # prova de contrato, não substitui VST DESeq2
    require(x_train and len(x_train)>1,'treino insuficiente')
    p=len(x_train[0]);require(all(len(x)==p for x in x_train),'dimensões inválidas')
    mu=[statistics.mean(col) for col in zip(*x_train)]
    sd=[statistics.stdev(col) or 1.0 for col in zip(*x_train)]
    return mu,sd

def apply_frozen_scaler(x,mu,sd):
    require(all(len(row)==len(mu) for row in x),'features incompatíveis')
    return [[(v-m)/s for v,m,s in zip(row,mu,sd)] for row in x]

def validate_budget(files,max_bytes=MAX_AGGREGATE_BYTES):
    ids=[r['file_id'] for r in files]
    require(len(ids)==len(set(ids)),'file_id duplicado; somatório ambíguo')
    require(all(r.get('access')=='open' and r.get('state')=='released' for r in files),'acesso controlado ou estado não-released')
    require(all(type(r.get('file_size')) is int and r['file_size']>0 for r in files),'file_size inválido')
    total=sum(r['file_size'] for r in files)
    require(total<=max_bytes,f'orçamento excedido: {total} > {max_bytes}')
    return total

def expect_red(fn,*args,**kwargs):
    try: fn(*args,**kwargs)
    except GateBlocked: return True
    return False

class R934SyntheticTDD(unittest.TestCase):
    def test_counts_green(self): self.assertTrue(validate_counts([[1,2],[3,1]]))
    def test_counts_nan_red(self): self.assertTrue(expect_red(validate_counts,[[1,float('nan')],[2,1]]))
    def test_counts_fraction_red(self): self.assertTrue(expect_red(validate_counts,[[1,1.5],[2,1]]))
    def test_zero_library_red(self): self.assertTrue(expect_red(validate_counts,[[1,0],[2,0]]))
    def test_patient_isolation_green(self): self.assertTrue(validate_patient_partition(['P1','P2'],['P3']))
    def test_patient_leakage_red(self): self.assertTrue(expect_red(validate_patient_partition,['P1','P2'],['P2','P3']))
    def test_frozen_parameters_green(self):
        mu,sd=fit_train_scaler([[1,10],[3,14],[5,18]])
        z=apply_frozen_scaler([[101,14]],mu,sd)
        self.assertAlmostEqual(z[0][0],(101-mu[0])/sd[0]); self.assertEqual(mu[0],3)
    def test_controlled_access_red(self):
        self.assertTrue(expect_red(validate_budget,[{'file_id':'A','access':'controlled','state':'released','file_size':10}]))
    def test_over_budget_red(self):
        self.assertTrue(expect_red(validate_budget,[{'file_id':'A','access':'open','state':'released','file_size':MAX_AGGREGATE_BYTES+1}]))
    def test_duplicate_file_red(self):
        f=[{'file_id':'X','access':'open','state':'released','file_size':100}]
        self.assertTrue(expect_red(validate_budget,f+f))
    def test_budget_green(self):
        self.assertEqual(validate_budget([{'file_id':'X','access':'open','state':'released','file_size':500}]),500)

buffer=io.StringIO()
suite=unittest.defaultTestLoader.loadTestsFromTestCase(R934SyntheticTDD)
result=unittest.TextTestRunner(stream=buffer,verbosity=2).run(suite)
TDD_GREEN = result.wasSuccessful()
write_json(ROOT/'tests'/'tdd_report.json',{'timestamp_utc':timestamp(),'tests':result.testsRun,
    'failures':[str(e) for _,e in result.failures], 'errors':[str(e) for _,e in result.errors],
    'green':TDD_GREEN,'scope':'synthetic preflight only','log':buffer.getvalue()})
print(buffer.getvalue())
require(TDD_GREEN,'P0 sintético: testes falharam')

## 3. Contrato de seleção do GDC — exclusivamente metadata-only

Consulta GET ao `https://api.gdc.cancer.gov/status`, GET ao `/files/_mapping` e paginação GET `/files` (não `/data`). No Colab, execute esta seção **somente se desejar atualização de metadados**. A enumeração anterior de 614/605 não é uma validação reproduzida pelo notebook.

O GDC pode mudar seu esquema; se um campo ou filtro falhar, **parar**, registrar e revisar. Não remover o filtro `access=open` para prosseguir. Os metadados brutos ficam em `private_metadata/` (não compartilhar). Acesso a dados individuais continua proibido.

In [ ]:
GDC_FIELDS=[
 'file_id','file_name','file_size','md5sum','access','state','data_category','data_type',
 'analysis.workflow_type','analysis.workflow_version','version','data_release',
 'cases.case_id','cases.submitter_id','cases.project.project_id',
 'cases.samples.sample_id','cases.samples.submitter_id','cases.samples.sample_type',
 'cases.samples.portions.analytes.aliquots.aliquot_id',
 'cases.samples.portions.analytes.aliquots.submitter_id']
GDC_FILTER={'op':'and','content':[
 {'op':'=','content':{'field':'cases.project.project_id','value':PROJECT}},
 {'op':'=','content':{'field':'files.data_category','value':'Transcriptome Profiling'}},
 {'op':'=','content':{'field':'files.data_type','value':'Gene Expression Quantification'}},
 {'op':'=','content':{'field':'files.analysis.workflow_type','value':EXPECTED_WORKFLOW}},
 {'op':'=','content':{'field':'files.access','value':'open'}},
 {'op':'=','content':{'field':'files.state','value':'released'}}]}

def gdc_get(path,params=None):
    import requests
    # Não há autenticação nem fallback para controlled-access.
    response=requests.get(GDC+path,params=params,timeout=90,headers={'Accept':'application/json'})
    response.raise_for_status(); payload=response.json()
    require(isinstance(payload,dict),'resposta GDC não é JSON-objeto')
    if 'errors' in payload: raise MetadataError(f'GDC retornou erros: {payload["errors"]}')
    return payload

def verify_files_schema(mapping):
    """Confronta filtros e identificadores críticos com /files/_mapping; falha fechada."""
    m=mapping.get('_mapping',{})
    require(isinstance(m,dict) and bool(m),'schema /files/_mapping ausente')
    # Exceções: version/data_release podem aparecer somente na resposta, não no índice de busca.
    indexed={
       'files.access','files.state','files.data_category','files.data_type',
       'files.analysis.workflow_type','files.analysis.workflow_version',
       'files.file_size','files.md5sum','cases.project.project_id',
       'cases.case_id','cases.submitter_id',
       'cases.samples.sample_type','cases.samples.sample_id',
       'cases.samples.portions.analytes.aliquots.aliquot_id',
       'cases.samples.portions.analytes.aliquots.submitter_id'
    }
    missing=[field for field in sorted(indexed)
             if field not in m and ('files.'+field) not in m]
    require(not missing,'campos não confirmados no schema GDC: '+', '.join(missing))
    return True

def metadata_pages(page_size=200):
    require(0<page_size<=1000,'page_size inválido')
    status=gdc_get('/status')
    mapping=gdc_get('/files/_mapping')
    verify_files_schema(mapping)
    # Map é arquivado integralmente e com hash para auditoria; não inferir campos faltantes.
    q={'filters':json.dumps(GDC_FILTER,separators=(',',':')),
       'fields':','.join(GDC_FIELDS),'format':'JSON','size':page_size,'from':0}
    rows=[]; page_audit=[]; expected_total=None; page_no=0
    while True:
        q['from']=len(rows)
        response=gdc_get('/files',dict(q))
        data=response.get('data',{})
        hits=data.get('hits')
        pag=data.get('pagination',{})
        require(isinstance(hits,list) and isinstance(pag.get('total'),int),
                'schema de paginação incompatível')
        if expected_total is None: expected_total=pag['total']
        require(pag['total']==expected_total,'total de paginação mudou durante a consulta')
        require(len(hits)>0 or len(rows)==expected_total,'página vazia antes do total')
        page_no+=1
        write_json(ROOT/'private_metadata'/f'gdc_raw_page_{page_no:04d}.json', response)
        page_audit.append({'page':page_no,'from':q['from'],'size':len(hits),'sha256':sha256_obj(response)})
        rows.extend(hits)
        require(len(rows)<=expected_total,'mais registros do que o total GDC')
        if len(rows)==expected_total: break
    ids=[r.get('file_id',r.get('id')) for r in rows]
    require(len(ids)==len(set(ids)),'file_id duplicados no retorno da API; consultar novamente')
    require(None not in ids,'file_id ausente')
    metadata={'retrieved_utc':timestamp(),'api':GDC,'status':status,
      'query_filter':GDC_FILTER,'fields_requested':GDC_FIELDS,
      'total_api':expected_total,'pages':page_audit,'schema_sha256':sha256_obj(mapping),
      'query_sha256':sha256_obj(GDC_FILTER),'records_sha256':sha256_obj(rows)}
    write_json(ROOT/'private_metadata'/'gdc-files-metadata-raw.json',{'audit':metadata,'files':rows})
    write_json(ROOT/'private_metadata'/'gdc-schema-mapping.json',mapping)
    return metadata,rows

## 4. Parsing estrito por caso, amostra e alíquota — fail-closed

**Uma linha por GDC file UUID** e **uma associação inequívoca** de caso, amostra e alíquota por arquivo. Se houver múltiplos vínculos ou identificadores faltantes, o arquivo é bloqueado e o manifesto exige reconciliação humana. Alíquotas múltiplas do mesmo paciente/condição são desempatas pelo barcode *antes* da análise, sem consultar expressão ou sobrevida.

In [ ]:
UUID_RE=re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')
MD5_RE=re.compile(r'^[0-9a-fA-F]{32}$')

def unique_one(values,what):
    values={str(x) for x in values if x not in (None,'')}
    if len(values)!=1: raise MetadataError(f'{what}: esperado identificador único; encontrados {len(values)}')
    return next(iter(values))

def unfold_file(record):
    file_id=record.get('file_id',record.get('id'))
    require(isinstance(file_id,str) and UUID_RE.fullmatch(file_id),'UUID GDC ausente/inválido')
    require(record.get('access')=='open' and record.get('state')=='released',
            f'Arquivo {file_id} não é open + released')
    require(record.get('data_category')=='Transcriptome Profiling' and
            record.get('data_type')=='Gene Expression Quantification','categoria/tipo inesperado')
    analysis=record.get('analysis') or {}
    require(analysis.get('workflow_type')==EXPECTED_WORKFLOW,'workflow inesperado')
    wf_version=analysis.get('workflow_version')
    require(bool(wf_version),'workflow_version ausente')
    size=record.get('file_size');require(type(size) is int and size>0,'tamanho inválido')
    md5=record.get('md5sum');require(isinstance(md5,str) and MD5_RE.fullmatch(md5) is not None,'MD5 inválido')
    cases=record.get('cases') or []
    case_id=unique_one([c.get('case_id') for c in cases],'case_id')
    patient=unique_one([c.get('submitter_id') for c in cases],'patient barcode')
    require(len(cases)==1,'arquivo associado a múltiplos casos')
    links=[]
    for c in cases:
        project=(c.get('project') or {}).get('project_id')
        require(project==PROJECT,'projeto diferente de TCGA-KIRC')
        for sample in c.get('samples') or []:
            for portion in sample.get('portions') or []:
                for analyte in portion.get('analytes') or []:
                    for aliquot in analyte.get('aliquots') or []:
                        links.append({'sample_id':sample.get('sample_id'),
                            'sample_barcode':sample.get('submitter_id'),
                            'sample_type':sample.get('sample_type'),
                            'aliquot_id':aliquot.get('aliquot_id'),
                            'aliquot_barcode':aliquot.get('submitter_id')})
    links=[dict(t) for t in {tuple(sorted(x.items())) for x in links}]
    require(len(links)==1,f'arquivo {file_id}: vínculos amostrais ausentes ou ambíguos')
    x=links[0]
    require(all(x.get(k) for k in ('sample_id','sample_type','aliquot_id','aliquot_barcode')),'metadados da alíquota incompletos')
    require(UUID_RE.fullmatch(str(x['sample_id'])) and UUID_RE.fullmatch(str(x['aliquot_id'])),'sample_id/aliquot_id inválidos')
    require(str(x['aliquot_barcode']).startswith(patient+'-'),'alíquota não corresponde ao paciente')
    require(x['sample_type'] in ('Primary Tumor','Solid Tissue Normal'),'tipo amostral não elegível')
    return dict(file_id=file_id,file_name=record.get('file_name'),file_size=size,md5sum=md5,
        access=record['access'],state=record['state'],workflow_version=wf_version,
        version=record.get('version'),data_release=record.get('data_release'),
        case_id=case_id,patient_id=patient,**x)

def build_manifest(metadata,records):
    require(len(records)==metadata['total_api'],'páginas insuficientes')
    file_ids=[r.get('file_id',r.get('id')) for r in records]
    require(len(set(file_ids))==metadata['total_api'],'total API ≠ IDs únicos')
    selected=[];excluded=[];ambiguous=[]
    for rec in records:
        fid=rec.get('file_id',rec.get('id'))
        # Amostras não tumor/normal podem ser excluídas antes da verificação das ligações profundas
        st={s.get('sample_type') for c in (rec.get('cases') or []) for s in (c.get('samples') or [])}
        if st and not st.intersection({'Primary Tumor','Solid Tissue Normal'}):
            excluded.append({'file_id':fid,'reason':'sample_type não elegível'});continue
        try: selected.append(unfold_file(rec))
        except GateBlocked as e:
            ambiguous.append({'file_id':fid,'reason':str(e)})
    # Falha fechada: qualquer vínculo ambíguo ou desconhecido impede o GREEN do manifesto.
    require(not ambiguous,f'{len(ambiguous)} arquivos sem vínculo auditável; registrar e revisar')
    require(bool(selected),'nenhum arquivo elegível')
    # Escolher 1 alíquota por patient_id e condição ANTES de inspecionar OS/expressão
    picked={}; duplicates=[]
    for row in sorted(selected,key=lambda r:(r['patient_id'],r['sample_type'],r['aliquot_barcode'],r['file_id'])):
        key=(row['patient_id'],row['sample_type'])
        if key in picked:
            duplicates.append({'file_id':row['file_id'],'reason':'alíquota elegível alternativa; seleção lexicográfica'})
        else: picked[key]=row
    chosen=list(picked.values())
    require(len({r['file_id'] for r in chosen})==len(chosen),'UUID duplicado após seleção')
    wf={str(r['workflow_version']) for r in chosen}
    require(len(wf)==1,f'workflow_version heterogêneo: {sorted(wf)}')
    # data_release pode vir como lista de releases por arquivo, não exige valor textual idêntico.
    release_sets=[]
    for row in chosen:
        raw=row['data_release']
        vals=raw if isinstance(raw,list) else [raw]
        vals={str(v) for v in vals if v not in (None,'')}
        require(bool(vals),f'data_release ausente: {row["file_id"]}')
        release_sets.append(vals)
    common_release=set.intersection(*release_sets)
    require(bool(common_release),'nenhum release GDC comum a todos os arquivos')
    require(all(r.get('version') not in (None,'') for r in chosen),'version por arquivo ausente')
    total=validate_budget(chosen)
    manifest={'created_at_utc':timestamp(),'status':'VALIDATED_METADATA_NOT_DOWNLOADED',
        'query_audit':metadata,'selected_count':len(chosen),'selection_pre_patient_dedup':len(selected),
        'excluded_file_types_count':len(excluded),'alternative_aliquots_count':len(duplicates),
        'total_bytes':total,'max_bytes':MAX_AGGREGATE_BYTES,
        'workflow_version':next(iter(wf)),'common_data_releases':sorted(common_release),
        'records':chosen,'excluded':excluded,'alternatives':duplicates}
    return manifest

In [ ]:
import copy

def fake_gdc_record():
    return {
      'file_id':'00000000-0000-0000-0000-000000000001','file_name':'fake.tsv',
      'file_size':200,'md5sum':'0'*32,'access':'open','state':'released',
      'data_category':'Transcriptome Profiling','data_type':'Gene Expression Quantification',
      'analysis':{'workflow_type':'STAR - Counts','workflow_version':'1.0'},
      'version':'1','data_release':['Test Release'],
      'cases':[{'case_id':'00000000-0000-0000-0000-000000000010',
                'submitter_id':'TCGA-AA-0001','project':{'project_id':'TCGA-KIRC'},
                'samples':[{'sample_type':'Primary Tumor',
                            'sample_id':'00000000-0000-0000-0000-000000000020',
                            'submitter_id':'TCGA-AA-0001-01A',
                            'portions':[{'analytes':[{'aliquots':[{
                               'aliquot_id':'00000000-0000-0000-0000-000000000030',
                               'submitter_id':'TCGA-AA-0001-01A-01R-0000-00'}]}]}]}]}]}

def check_metadata_parser():
    good=fake_gdc_record()
    assert unfold_file(good)['sample_type']=='Primary Tumor'
    bad=copy.deepcopy(good);bad['access']='controlled'
    assert expect_red(unfold_file,bad)
    bad=copy.deepcopy(good);bad['cases'][0]['samples'][0]['portions'][0]['analytes'][0]['aliquots'].append({
        'aliquot_id':'00000000-0000-0000-0000-000000000031',
        'submitter_id':'TCGA-AA-0001-01A-01R-0001-00'})
    assert expect_red(unfold_file,bad)
    bad=copy.deepcopy(good);bad['md5sum']='bad'
    assert expect_red(unfold_file,bad)
    met={'total_api':1,'retrieved_utc':timestamp()}
    m=build_manifest(met,[good])
    assert m['total_bytes']==200 and m['selected_count']==1
    bad=copy.deepcopy(good);bad['data_release']=None
    assert expect_red(build_manifest,met,[bad])
    return True
assert check_metadata_parser()
write_json(ROOT/'tests'/'metadata_parser_smoke.json',{'passed':True,'scope':'synthetic-only','timestamp_utc':timestamp()})
print('5 contraprovas e invariantes adicionais de metadados: PASS (sintético).')

## 5. Executar **somente** a consulta de metadados (opcional)

Para iniciar a etapa P1 no Colab, altere `CONSULTAR_GDC_METADATA = True` na seção 1, execute esta célula e examine os JSONs locais. Nenhum dado de expressão é baixado aqui. Falhas de schema ou orçamento **não** são contornadas. Se o fluxo parar, mantenha o relatório e solicite revisão.

In [ ]:
GDC_MANIFEST=None
if CONSULTAR_GDC_METADATA:
    metadata,records=metadata_pages(page_size=200)
    try:
        GDC_MANIFEST=build_manifest(metadata,records)
        write_json(ROOT/'private_metadata'/'gdc-manifest-R934.json',GDC_MANIFEST)
        print('P1: manifesto verificado em metadados (não significa autorização de download).')
        print({k:GDC_MANIFEST[k] for k in ('selected_count','total_bytes','workflow_version','common_data_releases')})
    except GateBlocked as e:
        write_json(ROOT/'logs'/'p1-blocker.json',{'time_utc':timestamp(),'reason':str(e)})
        print('P1 RED — bloqueado:',e)
else:
    print('Modo seguro: nenhuma requisição foi enviada ao GDC.')

## 6. Selo dos gates P0–P4 e registro da revisão humana

Os estados abaixo não podem ser promovidos por resultados biológicos, número de genes ativos, p-valores, AUC ou HR. P0 só será GREEN após um aceite metodológico **registrado pelo pesquisador** e verificação independente dos testes. P1 depende de manifesto validado, compatibilidade de workflow/release e confirmação humana de que o release comum está vigente. A aprovação dos metadados **não** comprova arquivos obtidos.

In [ ]:
# O aceite metodológico precisa de documento externo e hash, não apenas deste botão/variável.
REVISAO_REGISTRADA=False
REVISAO_DOCUMENTO_SHA256=''  # 64 caracteres hex de parecer metodológico revisado
RELEASE_VIGENTE_CONFERIDO=False  # comparado ao release GDC atual e documentado por revisor
P2_DOWNLOADS_VERIFICADOS=False
P3_PIPELINE_NESTED_VST_VERIFICADO=False
P4_ANALISE_REVISADA=False

def gates_snapshot():
    accepted=bool(REVISAO_REGISTRADA and ACEITE_METODOLOGICO.strip() and
                  re.fullmatch('[0-9a-fA-F]{64}',REVISAO_DOCUMENTO_SHA256))
    states={
      'P0': bool(TDD_GREEN and accepted and PANEL_APPROVED),
      'P1': bool(GDC_MANIFEST is not None and GDC_MANIFEST.get('status')=='VALIDATED_METADATA_NOT_DOWNLOADED' and RELEASE_VIGENTE_CONFERIDO),
      'P2': bool(P2_DOWNLOADS_VERIFICADOS),
      'P3': bool(P3_PIPELINE_NESTED_VST_VERIFICADO),
      'P4': bool(P4_ANALISE_REVISADA)
    }
    # Sequencial: não há GREEN posterior enquanto o anterior é RED.
    for i in range(1,5):
        k=f'P{i}'
        states[k]=bool(states[k] and states[f'P{i-1}'])
    return states
GATES=gates_snapshot()
write_json(ROOT/'reports'/'gates_R934.json',{'time_utc':timestamp(),'spec_hash':SPEC_HASH,'gates':GATES,
    'note':'TDD sintético != validação de dados TCGA; revisão independente exigida'})
print('Gates:',GATES)

## 7. Downloader opcional P2 — somente manifesto aprovado

**NÃO EXECUTE sem validação P0/P1 e revisão humana.** A transferência só pode ocorrer para UUIDs já incluídos no manifesto, todos open/released, com orçamento total ≤6 GiB. Cada arquivo é conferido pelo MD5 do GDC e por SHA-256 local. Não há autenticação, bypass ou download de controlled-access.

Por padrão `EXECUTAR_DOWNLOADS=False`. Depois dos gates, a pessoa responsável deve inserir `CONFIRMACAO_DOWNLOAD='AUTORIZO_DOWNLOAD_OPEN_6GIB'`. Arquivos inválidos **não são** aceitos silenciosamente; interrupção preserva evidências e aborta.

In [ ]:
def download_open_manifest(manifest,directory):
    import requests
    require(gates_snapshot().get('P0') and gates_snapshot().get('P1'),'P0/P1 não aprovados')
    require(EXECUTAR_DOWNLOADS,'download desabilitado')
    require(CONFIRMACAO_DOWNLOAD=='AUTORIZO_DOWNLOAD_OPEN_6GIB','confirmação explícita ausente')
    entries=manifest['records']; total=validate_budget(entries)
    require(total==manifest['total_bytes'],'manifesto e somatório divergentes')
    dest=Path(directory);dest.mkdir(parents=True,exist_ok=True)
    report=[];downloaded=0
    for item in entries:
        fid=item['file_id']
        require(UUID_RE.fullmatch(fid) is not None,'UUID inválido')
        destfile=dest/(fid+'.gdc')  # conteúdo bruto (formato real registrado no manifesto), UUID seguro
        partial=destfile.with_suffix('.partial')
        h_md5=hashlib.md5();h_sha=hashlib.sha256();count=0
        with requests.get(GDC+'/data/'+fid,stream=True,timeout=(20,180)) as response:
            response.raise_for_status()
            with partial.open('wb') as out:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    if not chunk: continue
                    count+=len(chunk)
                    require(count<=item['file_size'] and downloaded+count<=MAX_AGGREGATE_BYTES,
                            'transferência excedeu o manifesto/orçamento')
                    h_md5.update(chunk);h_sha.update(chunk);out.write(chunk)
        require(count==item['file_size'],'tamanho transferido não corresponde ao manifesto')
        require(h_md5.hexdigest().lower()==item['md5sum'].lower(),'checksum MD5 divergente')
        partial.replace(destfile)
        downloaded+=count
        report.append({'file_id':fid,'bytes':count,'md5':h_md5.hexdigest(),'sha256':h_sha.hexdigest()})
        write_json(ROOT/'private_metadata'/'download-progress.json',{'checked':report,'completed_bytes':downloaded})
    write_json(ROOT/'private_metadata'/'download-complete.json',{'finished_utc':timestamp(),
        'manifest_sha256':sha256_obj(manifest),'files':report,'downloaded_bytes':downloaded})
    return report

if EXECUTAR_DOWNLOADS:
    require(GDC_MANIFEST is not None,'sem manifesto completo e validado')
    _download_report=download_open_manifest(GDC_MANIFEST,ROOT/'data_open')
else:
    print('P2 desativado; nenhum arquivo RNA-seq foi baixado.')

## 8. P3 — desenho prognóstico e funções de proteção (não analisar dados TCGA ainda)

**Unidade = paciente.** Estratificar por evento (70% descoberta/30% holdout) com seed 2026. Registrar IDs e somente seus hashes em relatórios públicos. Não usar dados da validação para escolher genes, regularização, z-score, dispersão ou cutoff. PCK1/PCK2 pertencem à lista separada de gliconeogênese. Seleção LASSO deve usar CV aninhada 5×5, 5 repetições no discovery, e `lambda.1se` final, sem fallback para `lambda.min` em caso de modelo vazio.

O pipeline estatístico definitivo demanda **integração versionada R/Bioconductor (DESeq2/VST), `glmnet`/`survival` e `timeROC`** depois que P2 estiver completo. Este notebook implementa validação de entrada e partição sem executar modelos reais, evitando dar aparência de cálculo clínico.

In [ ]:
def make_stratified_split(records,seed=SEED):
    """records: lista de dicts com patient_id, event (0/1); depende de sklearn no Colab."""
    from sklearn.model_selection import train_test_split
    ids=[r['patient_id'] for r in records]
    ev=[r['event'] for r in records]
    require(len(ids)==len(set(ids)),'pacientes repetidos antes do particionamento')
    require(len(ids)>20 and set(ev)=={0,1},'amostra/eventos insuficientes')
    require(min(Counter(ev).values())>=10,'estratos com poucos pacientes')
    train,test=train_test_split(records,test_size=.30,random_state=seed,stratify=ev)
    validate_patient_partition([x['patient_id'] for x in train],[x['patient_id'] for x in test])
    return train,test

def stratified_fold_ids(records,k=5,seed=SEED):
    from sklearn.model_selection import StratifiedKFold
    events=[r['event'] for r in records]
    require(min(Counter(events).values())>=k,'evento/censura insuficiente para folds')
    splitter=StratifiedKFold(n_splits=k,shuffle=True,random_state=seed)
    ids={}
    for fold,(tr,va) in enumerate(splitter.split(range(len(records)),events)):
        require(bool(sum(events[i] for i in tr)) and bool(sum(events[i] for i in va)),
                f'fold {fold} sem eventos')
        for i in va: ids[records[i]['patient_id']]=fold
    require(len(ids)==len(records),'folds não cobriram todas as pessoas')
    return ids

# DEMONSTRAÇÃO SINTÉTICA, sem expressão humana e sem resultados de sobrevivência.
synthetic=[{'patient_id':f'FAKE-{i:03d}','event':int(i%3==0)} for i in range(150)]
train_demo,holdout_demo=make_stratified_split(synthetic)
folds_demo=stratified_fold_ids(train_demo,5,SEED)
assert len(train_demo)+len(holdout_demo)==150
write_json(ROOT/'tests'/'demo_synthetic_split.json',{'seed':SEED,'train_n':len(train_demo),
   'holdout_n':len(holdout_demo),'train_event_n':sum(r['event'] for r in train_demo),
   'holdout_event_n':sum(r['event'] for r in holdout_demo),
   'fold_counts':dict(Counter(folds_demo.values())),'note':'DADOS SINTÉTICOS, NÃO TCGA'})
print('Demonstração sintética do split OK. Nenhuma análise prognóstica real executada.')

### Especificação operacional do módulo R a desenvolver antes do P3 GREEN

**Entrada:** um *manifesto P2 verificado* e apenas dados open-access necessários. `patient_id` interno, contagens não negativas inteiras com Ensembl ID + gene annotation, tabela clínica de OS/idade/sexo/estágio. Para contagens duplicadas da mesma entidade, somar inteiros; nunca `limma::avereps`. Separar tumores primários (prognóstico) e normais (MUI descritivo) e registrar inclusão/exclusão. Descartar tempos OS ausentes/não positivos mediante fluxograma; não impor 30 dias sem justificativa.

**Pré-processamento:** filtros de CPM e geometria/size factors aprendidos no discovery (ou no treino do fold), dispersão VST e escala congeladas. Para holdout, não executar estimadores distributivos novos. **Modelagem:** 5 folds externos × 5 repetições; para cada outer-train, fazer 5 folds internos, com todas as transformações ajustadas só no train interno; a agregação out-of-fold descreve estabilidade, não seleciona o modelo final. Depois fazer CV 5-fold no discovery completo, escolher `lambda.1se` e treinar modelo clínico e ampliado. Se seleção resultar vazia, registrar isso sem trocar arbitrariamente de lambda ou genes.

**Avaliação:** holdout aberto uma vez; AUC cumulative/dynamic IPCW em 60 meses (se ≥10 eventos até 60 meses, censura Ĝ(60)≥0,10 e suporte de seguimento), bootstrap pareado B=1000 com ≥90% reamostragens estimáveis. O modelo de censura deve usar o discovery como referência, sem estimar IPW no holdout para o preditor. Reportar C-index, Brier e calibração separadamente. Sem resultados desejados como gates.

**Ressalva:** esta seção é especificação executável a implementar/validar no módulo R, não uma análise executada. P3 permanece RED.

## 9. Predição genômica (Genomic Intelligence) — evidência separada

Mantenha as predições preliminares dos genes **SLC2A1** e **SDHB** como resultados computacionais exploratórios, sem vinculá-los ao risco clínico. Os resultados do plugin são distintos do GDC e não serão usados na seleção de características a partir do holdout. O script inicial do estudo contém números não verificados; não incorporá-los às tabelas finais.

In [ ]:
GI_PILOT=[{'gene':'SLC2A1','ensembl_id':'ENSG00000117394','pred_promoter_windows':13,
    'pred_tpm':39.9449,'reference':'Ensembl/GRCh38','type':'in_silico_not_TCGA'},
   {'gene':'SDHB','ensembl_id':'ENSG00000117118','pred_promoter_windows':7,
    'pred_tpm':126.9445,'reference':'Ensembl/GRCh38','type':'in_silico_not_TCGA'}]
write_json(ROOT/'reports'/'genomic_intelligence_exploratory.json',{
 'source':'Genomic Intelligence plugin, execução preliminar na conversa anterior',
 'limits':'Não representa diferença tumor-normal, coorte TCGA ou validação clínica',
 'results':GI_PILOT})
print('Genomic Intelligence: predições isoladas do pipeline de prognóstico.')

## 10. Auditoria, integridade e exportação local

Gera hashes de arquivos de relatório **sem incluir dados individuais**. Arquivos privados de metadados, IDs de pacientes e resultados individuais não entram no pacote compartilhável. A execução no Colab poderá produzir resultados em `/content/R934_ccRCC/`. É responsabilidade do operador proteger qualquer dado individual mesmo quando open-access.

In [ ]:
import zipfile
PUBLIC_DIRS=['tests','reports','logs']
public_files=[p for folder in PUBLIC_DIRS for p in (ROOT/folder).rglob('*') if p.is_file()]
checksum=[{'path':str(p.relative_to(ROOT)),'size':p.stat().st_size,
    'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in public_files]
write_json(ROOT/'reports'/'public_artifact_checksums.json',checksum)
zip_path=ROOT/'R934_auditoria_publica.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in ROOT.glob('tests/**/*'):
        if p.is_file(): zf.write(p,p.relative_to(ROOT))
    for p in ROOT.glob('reports/**/*'):
        if p.is_file(): zf.write(p,p.relative_to(ROOT))
    for p in ROOT.glob('logs/**/*'):
        if p.is_file(): zf.write(p,p.relative_to(ROOT))
print('Relatório ZIP local:',zip_path)
print('Estado ao encerrar:',gates_snapshot())

## 11. Limites da implementação e referências

**Executável agora no Colab:** integridade sintética e contraprovas, congelamento SDD, documentação/hash, chamada opcional a `/status`, `/files/_mapping` e `/files` (metadata-only), parsing e validação do manifesto, bloqueios de orçamento, módulo de download controlado por consentimento e testes de split (sintéticos).

**Ainda não executado e não aprovado:** download de contagens, auditoria clínica, transformação VST/R, nested LASSO-Cox, AUC com IC 95%, Brier, nomograma/calibração, análises imunes/ssGSEA e validação externa. P2–P4 permanecem RED até a implementação estatística efetiva e a revisão humana. A AUC 0,79 vs 0,69 ou outros números antecipados no manuscrito não são resultados observados.

**Documentação consultável:**
- National Cancer Institute, GDC API — [Getting Started](https://docs.gdc.cancer.gov/API/Users_Guide/Getting_Started/) e [System Information](https://docs.gdc.cancer.gov/API/Users_Guide/System_Information/).
- GDC Search API: [Search and Retrieval](https://docs.gdc.cancer.gov/API/Users_Guide/Search_and_Retrieval/).
- Scikit-survival (para futura avaliação IPCW): [cumulative_dynamic_auc](https://scikit-survival.readthedocs.io/en/stable/api/generated/sksurv.metrics.cumulative_dynamic_auc.html).
- Documento fornecido: *Protocolo pré-análise R934 — ccRCC / TCGA-KIRC*, 21/10/2026 (base principal). Entendimentos divergentes do documento anterior foram indicados explicitamente neste caderno.

**Segurança:** pesquisa retrospectiva exploratória; não é dispositivo médico nem orientação individual de tratamento. A permissão de aquisição cobre apenas arquivos `open` do GDC, com limite total previamente aprovado de 6 GiB e gates satisfeitos.